# PCB temperature prediction with Transolver
Source: [Multi-topology Dataset](https://github.com/henkuailederen/Multi-topology-Dataset), by Ruiting Ke, Jianfeng Tao and Chengliang Liu (CC BY 4.0). Cite the repository and accompanying study when using these data.

Predict the steady-state temperature field in °C from coordinates, power density, chip occupancy, ceramic/baseplate/copper masks and cooling. The supplied `max_temps` are labels and are never model inputs. This uses the standard Transolver physics-attention architecture adapted from this project's elasticity model.

**Kaggle setup:** attach the full dataset (retaining `TrainData`, `ValData`, `TestData`) and these two Python modules as Kaggle inputs. Enable a GPU. Set `DATA_ROOT` and `MODULE_DIR` below if automatic discovery does not match your uploaded directory structure. Outputs go to `/kaggle/working/pcb_outputs`. Dependencies: torch, numpy, h5py, matplotlib (normally preinstalled); install missing ones with `%pip install h5py matplotlib`. No data are downloaded by this notebook.

The local `samples/` directory contains only eight real samples for integration checks, and is not a benchmark. Preserve official splits; layouts can overlap across supplied splits, so these results do not establish generalization to unseen layouts.

In [ ]:
from pathlib import Path
import os, sys, json, random, time
import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

SMOKE = os.environ.get("PCB_SMOKE_TEST", "0") == "1"
KAGGLE = Path("/kaggle/input").exists()
MODULE_DIR = None  # e.g. Path('/kaggle/input/pcb-code/pcb')
DATA_ROOT = None   # directory immediately containing TrainData/ValData/TestData
if MODULE_DIR is None:
    candidates = [Path.cwd(), Path.cwd() / "pcb"]
    if KAGGLE:
        candidates += [p.parent for p in Path('/kaggle/input').rglob('dataset.py')
                       if 'class PCBDataset' in p.read_text(errors='ignore')]
    MODULE_DIR = next((p for p in candidates if (p / 'dataset.py').exists()
                       and (p / 'transolver.py').exists()), None)
if MODULE_DIR is None:
    raise FileNotFoundError('Attach pcb/dataset.py and pcb/transolver.py and set MODULE_DIR.')
sys.path.insert(0, str(MODULE_DIR.resolve()))
from dataset import PCBDataset, NormalizationStats, discover_files
from transolver import Transolver
if DATA_ROOT is None:
    candidates = ([p.parent for p in Path('/kaggle/input').rglob('TrainData') if p.is_dir()]
                  if KAGGLE else [MODULE_DIR / 'samples'])
    candidates = [p for p in candidates if all((p / s).is_dir() for s in
                                              ('TrainData', 'ValData', 'TestData'))]
    if len(candidates) != 1:
        raise ValueError(f'Set DATA_ROOT explicitly; found {candidates}')
    DATA_ROOT = candidates[0]
DATA_ROOT = Path(DATA_ROOT)
OUTPUT_DIR = Path('/kaggle/working/pcb_outputs') if KAGGLE else MODULE_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Modules:', MODULE_DIR, '\nData:', DATA_ROOT, '\nOutputs:', OUTPUT_DIR)

In [ ]:
CONFIG = dict(seed=42, epochs=1 if SMOKE else 100, batch_size=1 if SMOKE else 2,
              stride=4, lr=1e-3, weight_decay=1e-5, max_grad_norm=1.0,
              num_workers=0 if SMOKE or not KAGGLE else 2,
              amp=True, n_hidden=32 if SMOKE else 128,
              n_layers=2 if SMOKE else 8, n_heads=4 if SMOKE else 8,
              slice_num=8 if SMOKE else 32, mlp_ratio=1, dropout=0.0)
random.seed(CONFIG['seed']); np.random.seed(CONFIG['seed']); torch.manual_seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if SMOKE:
    torch.set_num_threads(2)
print(device, CONFIG)

`stride=4` samples every fourth pixel: 64×64 = 4,096 points. This reduces memory and runtime but may miss fine chip boundaries. Set `stride=1` for the native 256×256 field (65,536 points); reduce batch size first. All reported validation/test metrics below use the configured resolution. Coordinates retain their positions on the original grid. Input power and output temperature statistics are fitted exclusively on full-resolution training fields and saved with the checkpoint.

In [ ]:
train_files = discover_files(DATA_ROOT, 'TrainData')
val_files = discover_files(DATA_ROOT, 'ValData')
test_files = discover_files(DATA_ROOT, 'TestData')
if SMOKE:
    train_files, val_files, test_files = train_files[:4], val_files[:2], test_files[:2]
print('Fitting normalization on', len(train_files), 'training files...')
stats = NormalizationStats.fit(train_files)
train_ds = PCBDataset(DATA_ROOT, 'TrainData', stats, CONFIG['stride'], train_files)
val_ds = PCBDataset(DATA_ROOT, 'ValData', stats, CONFIG['stride'], val_files)
test_ds = PCBDataset(DATA_ROOT, 'TestData', stats, CONFIG['stride'], test_files)
def loader(ds, shuffle=False):
    return DataLoader(ds, batch_size=CONFIG['batch_size'], shuffle=shuffle,
                      num_workers=CONFIG['num_workers'], pin_memory=device.type == 'cuda',
                      persistent_workers=CONFIG['num_workers'] > 0)
train_loader, val_loader, test_loader = loader(train_ds, True), loader(val_ds), loader(test_ds)
print('Split sizes:', len(train_ds), len(val_ds), len(test_ds))
print('Item shapes:', [tuple(a.shape) for a in train_ds[0]], stats)

In [ ]:
model_kwargs = {k: CONFIG[k] for k in ('n_hidden', 'n_layers', 'n_heads', 'slice_num', 'mlp_ratio', 'dropout')}
model_kwargs.update(space_dim=2, fun_dim=6, out_dim=1)
model = Transolver(**model_kwargs).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])
amp_enabled = CONFIG['amp'] and device.type == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled, init_scale=1024.0)
print('Parameters:', sum(p.numel() for p in model.parameters()))

def run_epoch(loader, training=False):
    model.train(training)
    squared_error = absolute_error = normalized_error = 0.0
    count = 0
    with torch.set_grad_enabled(training):
        for coords, features, target in loader:
            coords, features, target = [a.to(device, non_blocking=True) for a in (coords, features, target)]
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=amp_enabled):
                prediction = model(coords, features)
                loss = torch.nn.functional.mse_loss(prediction.float(), target)
            if not torch.isfinite(loss):
                raise FloatingPointError('Nonfinite loss; inspect inputs or disable AMP.')
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'], error_if_nonfinite=not amp_enabled)
                scaler.step(optimizer)
                scaler.update()
            error = (prediction.detach().float() - target) * stats.temperature_std
            squared_error += error.square().sum().item()
            absolute_error += error.abs().sum().item()
            normalized_error += loss.item() * target.numel()
            count += target.numel()
    return dict(mse_normalized=normalized_error / count, mae_c=absolute_error / count,
                rmse_c=(squared_error / count)**0.5)

In [ ]:
history, best_val = [], float('inf')
for epoch in range(1, CONFIG['epochs'] + 1):
    start = time.time()
    train_metrics = run_epoch(train_loader, training=True)
    val_metrics = run_epoch(val_loader)
    scheduler.step()
    history.append(dict(epoch=epoch, train=train_metrics, val=val_metrics))
    state = dict(epoch=epoch, model=model.state_dict(), model_kwargs=model_kwargs,
                 optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                 scaler=scaler.state_dict(), normalization=stats.to_dict(), config=CONFIG,
                 val_metrics=val_metrics)
    torch.save(state, OUTPUT_DIR / 'last.pt')
    if val_metrics['rmse_c'] < best_val:
        best_val = val_metrics['rmse_c']
        torch.save(state, OUTPUT_DIR / 'best.pt')
    (OUTPUT_DIR / 'history.json').write_text(json.dumps(history, indent=2))
    print(f"Epoch {epoch:03d} | train RMSE {train_metrics['rmse_c']:.3f} C | "
          f"val RMSE {val_metrics['rmse_c']:.3f} C | {time.time()-start:.1f}s")

Select the checkpoint using validation only. Evaluate the test split once after training. Checkpoints include architecture settings, normalization, optimizer, scheduler, and AMP scaler state. Local sample metrics only check that the pipeline runs; they do not measure model quality.

In [ ]:
checkpoint = torch.load(OUTPUT_DIR / 'best.pt', map_location=device, weights_only=True)
model.load_state_dict(checkpoint['model'])
stats = NormalizationStats(**checkpoint['normalization'])
test_metrics = run_epoch(test_loader)
(OUTPUT_DIR / 'test_metrics.json').write_text(json.dumps(test_metrics, indent=2))
print('Best epoch:', checkpoint['epoch'], 'test:', test_metrics)
plt.figure(figsize=(7, 3))
plt.plot([r['train']['rmse_c'] for r in history], label='Train')
plt.plot([r['val']['rmse_c'] for r in history], label='Validation')
plt.xlabel('Epoch index'); plt.ylabel('RMSE (°C)'); plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'learning_curve.png', dpi=140); plt.show()

In [ ]:
# Visualize a held-out sample at the configured grid resolution.
model.eval()
coords, features, target = test_ds[0]
with torch.no_grad():
    prediction = model(coords[None].to(device), features[None].to(device))[0].float().cpu()
true_c = stats.decode_temperature(target).numpy().reshape(len(torch.unique(coords[:, 1])), -1)
pred_c = stats.decode_temperature(prediction).numpy().reshape(true_c.shape)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
lo, hi = min(true_c.min(), pred_c.min()), max(true_c.max(), pred_c.max())
for ax, a, title in zip(axes, [true_c, pred_c, np.abs(pred_c - true_c)],
                       ['Reference (°C)', 'Prediction (°C)', 'Absolute error (°C)']):
    im = ax.imshow(a, origin='upper', **({} if 'error' in title else dict(vmin=lo, vmax=hi)))
    ax.set_title(title); ax.axis('off'); fig.colorbar(im, ax=ax, shrink=0.75)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'test_prediction.png', dpi=140); plt.show()
np.savez_compressed(OUTPUT_DIR / 'test_prediction.npz', reference_c=true_c, prediction_c=pred_c)

To run inference at native resolution, construct another `PCBDataset(DATA_ROOT, 'TestData', stats, stride=1)` and pass its coordinate/features tensors to the same model under `torch.no_grad()`. Physics attention aggregates globally, so process all points together: independent spatial chunks change the prediction. Resolution transfer is supported by the architecture but its accuracy must be evaluated.

For local integration testing set `PCB_SMOKE_TEST=1` before running the notebook. This uses a smaller two-layer model and one epoch on at most 4/2/2 samples. On Kaggle leave it unset for the full configuration. The notebook can also run on CPU, though full training is intended for a GPU.